# CENP-A CUT&Tag Enrichment at centroAnno Repeats — Figure Notebook

**Purpose:** Reproduce all main-text figures from the V2 CENP-A enrichment pipeline.

**Key result:** All 30/30 chromosomes show CENP-A signal above background at centroAnno-predicted centromeric repeats. Empirical P values derived from chromosome- and length-matched shuffle iterations with Benjamini-Hochberg correction.

**Panels:**
- **Karyotype with CENP-A signal tracks** — chromosome ideograms with BigWig signal bars (above), CENP-A enriched satellite arrays (middle below), and all centroAnno satellite arrays (bottom below); 30 chromosomes
- **Chromosome-level enrichment** — beeswarm + boxplot per chromosome, with empirical significance from shuffle iterations
- **Array-centered metaprofile** — normalized CUT&Tag signal around CENP-A enriched satellite arrays (±1 Mb)
- **Per-array enrichment** — foreground vs local flank background scatter

**What are CENP-A enriched satellite arrays?**
These are centroAnno-predicted HOR satellite repeats that also show CENP-A binding above the chromosome median. centroAnno predicts centromeric repeat monomers genome-wide by sequence similarity. We flag intervals where mean log2(CENP-A signal) exceeds the per-chromosome median, then merge flagged intervals within 250 kb into contiguous arrays. This jointly uses sequence (centroAnno) and functional (CENP-A) evidence to narrow down candidate centromeric regions. There are 492 arrays across 30 chromosomes. These arrays are the unit of analysis: CENP-A signal is quantified within each array and compared to flanking regions and shuffled backgrounds.

**Signal quantification:** CUT&Tag is a chromatin profiling assay that maps protein–DNA interactions genome-wide via targeted Tn5 transposase cleavage. For each genomic interval, paired-end CUT&Tag fragments are counted and normalized by interval length and library size (fragments per kilobase per million mapped fragments), then log₂-transformed with a pseudocount. We refer to the resulting values as **normalized CUT&Tag signal** throughout. This normalization accounts for differences in region size and sequencing depth, but unlike RNA-seq the underlying counts reflect chromatin-bound CENP-A (or H3K27ac), not transcript abundance.

**Environment:** `conda activate r-visualizations`

**Last updated:** 2026-07-24

In [ ]:
# ============================================================================
# Setup: Libraries, paths, constants
# ============================================================================
suppressPackageStartupMessages({
  library(ggplot2)
  library(dplyr)
  library(tidyr)
  library(data.table)
  library(patchwork)
  library(scales)
})

# Paths
BASE_DIR <- "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/figure/cenpa-cuttag-enrichment"
DATA_DIR <- file.path(BASE_DIR, "data")
COUNTS_DIR <- file.path(DATA_DIR, "counts")
DOMAIN_DIR <- file.path(DATA_DIR, "domains")
DOMAIN_COUNTS_DIR <- file.path(DATA_DIR, "domain_counts")
DOMAIN_DISTANCE_DIR <- file.path(DATA_DIR, "domain_distance_profiles")
QC_DIR <- file.path(DATA_DIR, "qc")
RESULTS_DIR <- file.path(BASE_DIR, "results")

# Constants
CHROMOSOMES <- paste0("chr", c(1:28, "X", "Y"))

# Sample metadata
SAMPLES <- c("XG_150", "XG_151", "XG_152", "XG_153")
SAMPLE_LABELS <- c(
  "XG_150" = "CENP-A rep1",
  "XG_151" = "CENP-A rep2",
  "XG_152" = "H3K27ac",
  "XG_153" = "H3K27ac rep2"
)
SAMPLE_COLORS <- c(
  "XG_150" = "#2166AC",
  "XG_151" = "#92C5DE",
  "XG_152" = "#B2182B",
  "XG_153" = "#D6604D"
)

# ggplot2 theme
theme_cenpa <- theme_bw(base_size = 10) +
  theme(
    panel.grid.minor = element_blank(),
    panel.grid.major = element_line(linewidth = 0.2, color = "grey90"),
    strip.background = element_rect(fill = "grey95", color = "grey80"),
    strip.text = element_text(size = 9, face = "bold"),
    axis.text = element_text(color = "black"),
    legend.position = "bottom",
    plot.title = element_text(size = 11, face = "bold"),
    plot.subtitle = element_text(size = 9, color = "grey40")
  )

cat_colors <- c(
  "Strong enrichment" = "#2166AC",
  "Moderate enrichment" = "#92C5DE",
  "Weak enrichment" = "#F4A582"
)

message("Setup complete.")

---
## centroAnno Inter-Interval Gap Distribution

**Why this matters:** centroAnno predicts centromeric HOR monomers genome-wide
by sequence similarity. Adjacent monomers within the same satellite array are
separated by tiny gaps (median **223 bp**), while distinct arrays on the same
chromosome are separated by large gaps. The 700 bp threshold (dashed line)
separates intra-array gaps from inter-array gaps.

**Prediction:** Intervals separated by gaps < 700 bp (tight HOR clusters)
should show stronger CENP-A enrichment than those separated by larger gaps
(isolated or distantly-spaced monomers).



In [ ]:
# ============================================================================
# Inter-interval gap distribution of centroAnno HOR predictions
#
# Shows the bimodal gap distribution. The 700 bp threshold separates
# intra-array HOR monomers (tight clusters) from inter-array gaps.
# ============================================================================

suppressPackageStartupMessages({
  library(data.table)
  library(ggplot2)
  library(scales)
})

# --------------------------------------------------------------------------
# Read centroAnno intervals
# --------------------------------------------------------------------------
centroanno_file <- file.path(
  "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj",
  "output/outputs-from-centraAnno/hifiasm-0414/cautils-chrOnly",
  "repeat_regions.bed"
)

centro <- fread(centroanno_file, header = FALSE, select = 1:3,
                col.names = c("chrom", "start", "end"))
setorder(centro, chrom, start)

# Compute gaps between adjacent intervals on the same chromosome
centro[, gap := start - data.table::shift(end), by = chrom]
gaps <- centro[!is.na(gap) & gap >= 0, .(chrom, gap)]

n_total <- nrow(centro)
n_gaps  <- nrow(gaps)
gap_median <- median(gaps$gap)
frac_lt_700bp <- mean(gaps$gap < 700)
frac_lt_10kb  <- mean(gaps$gap < 10e3)
frac_gt_250kb  <- mean(gaps$gap > 250e3)

message(sprintf("centroAnno intervals: %d, gaps: %d, median gap: %.0f bp", n_total, n_gaps, gap_median))
cat(sprintf("Gaps < 700 bp: %.1f%%\n", 100 * frac_lt_700bp))
cat(sprintf("Gaps < 10 kb:  %.1f%%\n", 100 * frac_lt_10kb))
cat(sprintf("Gaps > 250 kb: %.1f%%\n", 100 * frac_gt_250kb))

# --------------------------------------------------------------------------
# Gap histogram (log10 x-axis) with 700 bp threshold
# --------------------------------------------------------------------------

p_gap_hist <- ggplot(gaps, aes(x = gap + 1)) +
  geom_histogram(
    bins = 80,
    fill = "grey65",
    color = "grey40",
    linewidth = 0.15
  ) +
  geom_vline(
    xintercept = 700,
    linetype = "dashed",
    linewidth = 0.8,
    color = "#2166AC"
  ) +
  annotate(
    "text",
    x = 1000, y = Inf,
    label = "700 bp",
    angle = 90,
    hjust = 1.1, vjust = 0.5,
    size = 3.8,
    color = "#2166AC",
    fontface = "bold"
  ) +
  scale_x_log10(
    breaks = trans_breaks("log10", function(x) 10^x),
    labels = trans_format("log10", math_format(10^.x)),
    expand = expansion(mult = c(0.01, 0.05))
  ) +
  scale_y_continuous(expand = expansion(mult = c(0, 0.05))) +
  annotation_logticks(sides = "b", outside = FALSE, linewidth = 0.2,
                      short = unit(0.5, "mm"), mid = unit(0.8, "mm"),
                      long = unit(1.2, "mm")) +
  labs(
    x = "Gap between adjacent centroAnno intervals (bp, log10)",
    y = "Count",
    title = "Bimodal gap distribution with 700 bp threshold",
    subtitle = sprintf(
      "%d intervals  ·  %d gaps  ·  median = %.0f bp  ·  %.0f%% < 700 bp  ·  %.0f%% > 250 kb",
      n_total, n_gaps, gap_median, 100 * frac_lt_700bp, 100 * frac_gt_250kb
    )
  ) +
  theme_cenpa +
  theme(
    plot.title = element_text(size = 12, face = "bold"),
    plot.subtitle = element_text(size = 9, color = "grey40"),
    axis.title.x = element_text(size = 10),
    panel.grid.minor = element_blank()
  )

print(p_gap_hist)

# Save
ggsave(
  filename = file.path("supp_centroAnno_gap_distribution.pdf"),
  plot = p_gap_hist,
  width = 8,
  height = 5,
  units = "in",
  device = cairo_pdf
)

ggsave(
  filename = file.path("supp_centroAnno_gap_distribution.png"),
  plot = p_gap_hist,
  width = 8,
  height = 5,
  units = "in",
  dpi = 600,
  bg = "white"
)

cat("\nSaved: supp_centroAnno_gap_distribution.pdf / .png\n")




In [ ]:
# ============================================================================
# Load all data
# ============================================================================

# -- Library sizes --
lib_file <- file.path(QC_DIR, "library_sizes_fragments.txt")
if (file.exists(lib_file)) {
  lib_sizes_dt <- fread(lib_file, header = FALSE, col.names = c("sample", "n_fragments"))
  lib_sizes <- setNames(lib_sizes_dt$n_fragments, lib_sizes_dt$sample)
} else {
  # Fallback
  lib_sizes <- c("XG_150" = 27156788, "XG_151" = 25305681,
                 "XG_152" = 30805412, "XG_153" = 111250740)
}
message("Library sizes:")
print(lib_sizes)

# -- Merged domains --
domains_file <- file.path(DOMAIN_DIR, "merged_domains_d250000.bed")
domains <- fread(domains_file, header = FALSE,
                 col.names = c("chrom", "start", "end", "domain_id", "size"))
message("Merged domains: ", nrow(domains))

# -- Domain counts (foreground signal at each domain) --
load_domain_counts <- function(sample) {
  f <- file.path(DOMAIN_COUNTS_DIR, paste0(sample, "_domains.txt"))
  if (!file.exists(f)) return(NULL)
  dt <- fread(f, header = FALSE,
              col.names = c("chrom", "start", "end", "domain_id", "size", "count"))
  dt[, sample := sample]
  dt[, length := end - start]
  return(dt)
}
domain_counts <- rbindlist(lapply(SAMPLES, load_domain_counts))
message("Domain counts: ", nrow(domain_counts) / length(SAMPLES), " domains per sample")

# -- Domain flank counts (local background) --
load_domain_flanks <- function(sample) {
  f <- file.path(DOMAIN_COUNTS_DIR, paste0(sample, "_domain_flanks.txt"))
  if (!file.exists(f)) return(NULL)
  dt <- fread(f, header = FALSE,
              col.names = c("chrom", "start", "end", "flank_id", "count"))
  dt[, sample := sample]
  dt[, length := end - start]
  return(dt)
}
domain_flanks <- rbindlist(lapply(SAMPLES, load_domain_flanks))
message("Domain flank counts: ", nrow(domain_flanks), " regions")

# -- Bg1 (chromosome-shuffled background) --
load_bg1 <- function(sample) {
  f <- file.path(COUNTS_DIR, paste0(sample, "_bg1_chrom_shuffle.txt"))
  if (!file.exists(f)) return(NULL)
  dt <- fread(f, header = FALSE,
              col.names = c("chrom", "start", "end", "iter_id", "interval_id", "count"))
  dt[, sample := sample]
  dt[, length := end - start]
  return(dt)
}
bg1 <- rbindlist(lapply(SAMPLES, load_bg1))
message("Bg1 (chromosome shuffle): ", nrow(bg1) / length(SAMPLES), " intervals per sample")

# -- Domain distance profiles (for metaprofile plot) --
load_domain_distance <- function(sample) {
  f <- file.path(DOMAIN_DISTANCE_DIR, paste0(sample, "_domain_bin_counts.txt"))
  if (!file.exists(f)) return(NULL)
  dt <- fread(f, header = FALSE,
              col.names = c("chrom", "start", "end", "domain_id", "bin_label", "count"))
  dt[, sample := sample]
  dt[, length := end - start]
  return(dt)
}
domain_dist <- rbindlist(lapply(SAMPLES, load_domain_distance))
message("Domain distance profiles loaded: ", nrow(domain_dist), " rows")

message("All data loaded.")

In [ ]:
# ============================================================================
# Normalize CUT&Tag signal + log2 transform with pseudocount
#
# CUT&Tag fragment counts are normalized by interval length (kb) and
# library size (millions of mapped fragments) to produce a fragment
# density metric comparable across regions and samples. This is analogous
# to the fragments-per-kilobase-per-million (FPKM) normalization used in
# RNA-seq, but the underlying data reflect chromatin immunoprecipitation
# (CUT&Tag) rather than transcript abundance. We refer to the result as
# "normalized CUT&Tag signal" throughout.
# ============================================================================
normalize_signal <- function(dt, lib_sizes) {
  dt[, norm_signal := (count / (length / 1000)) / (lib_sizes[sample] / 1e6)]
  return(dt)
}

domain_counts <- normalize_signal(domain_counts, lib_sizes)
domain_flanks <- normalize_signal(domain_flanks, lib_sizes)
bg1 <- normalize_signal(bg1, lib_sizes)
domain_dist <- normalize_signal(domain_dist, lib_sizes)

# Pseudocount: half the minimum non-zero normalized signal
all_signal <- c(domain_counts$norm_signal, domain_flanks$norm_signal,
                bg1$norm_signal, domain_dist$norm_signal)
min_nonzero <- min(all_signal[all_signal > 0], na.rm = TRUE)
pseudocount <- min_nonzero / 2
message("Min non-zero normalized signal: ", format(min_nonzero, digits = 3))
message("Pseudocount: ", format(pseudocount, digits = 3))
message("Note: log2(signal) < 0 when normalized signal < 1 — normal for sparse 1-kb bins")

domain_counts[, log2_signal := log2(norm_signal + pseudocount)]
domain_flanks[, log2_signal := log2(norm_signal + pseudocount)]
bg1[, log2_signal := log2(norm_signal + pseudocount)]
domain_dist[, log2_signal := log2(norm_signal + pseudocount)]

message("Normalization complete.")

---## CENP-A Enrichment by Gap Class: Tight Clusters vs. Isolated Intervals**Hypothesis:** centroAnno intervals separated by small gaps (< 700 bp) aretightly clustered HOR monomers within the centromeric satellite array — theseshould bind CENP-A. Intervals with large gaps (≥ 700 bp) to their neighborsare isolated or distantly-spaced monomers — these should have lower or noCENP-A enrichment.**Classification:** Each centroAnno interval is classified by the gaps to itsadjacent neighbors on the same chromosome:- **Tight**: gap to previous < 700 bp OR gap to next < 700 bp- **Isolated**: gaps on both sides ≥ 700 bp (or at chromosome ends)**Null model:** Within each chromosome, shuffle the tight/isolated labelsacross intervals. Compute the difference in median CENP-A signal (tight −isolated) for each shuffle iteration. The observed delta is compared againstthis null distribution.

In [ ]:
# ============================================================================
# CENP-A signal in tight-cluster vs isolated centroAnno intervals
# (normalized to genome-wide chromosome-mean background)
# ============================================================================

suppressPackageStartupMessages({
  library(data.table)
  library(ggplot2)
})

GAP_THRESHOLD <- 700L
PSEUDO <- 1e-3

# --------------------------------------------------------------------------
# Step 1: Chromosome-mean CENP-A background from genome-wide 25-kb bins
# --------------------------------------------------------------------------
topwin <- fread(file.path(RESULTS_DIR, "top_window_signal_fraction.csv"))
chr_bg <- topwin[, .(chrom, sample, chr_mean_raw = total_signal / n_bins)]
chr_bg <- dcast(chr_bg, chrom ~ sample, value.var = "chr_mean_raw")
setnames(chr_bg, c("XG_150", "XG_151"), c("chr_mean_150", "chr_mean_151"))

# --------------------------------------------------------------------------
# Step 2: Read centroAnno intervals and classify by gap
# --------------------------------------------------------------------------
centroanno_file <- file.path(
  "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj",
  "output/outputs-from-centraAnno/hifiasm-0414/cautils-chrOnly",
  "repeat_regions.bed"
)

centro <- fread(centroanno_file, header = FALSE, select = 1:3,
                col.names = c("chrom", "start", "end"))
setorder(centro, chrom, start)

centro[, next_gap := data.table::shift(start, type = "lead") - end, by = chrom]
centro[, prev_gap := start - data.table::shift(end, type = "lag"), by = chrom]

centro[, gap_class := ifelse(
  (!is.na(next_gap) & next_gap < GAP_THRESHOLD) |
  (!is.na(prev_gap) & prev_gap < GAP_THRESHOLD),
  "tight", "isolated"
)]

centro[, interval_key := paste(chrom, start, end, sep = "_")]

n_tight <- sum(centro$gap_class == "tight")
n_isolated <- sum(centro$gap_class == "isolated")
cat(sprintf("Tight intervals:  %d (%.1f%%)
", n_tight, 100 * n_tight / nrow(centro)))
cat(sprintf("Isolated intervals: %d (%.1f%%)
", n_isolated, 100 * n_isolated / nrow(centro)))

# --------------------------------------------------------------------------
# Step 3: Load CENP-A signal and normalize to chromosome-mean background
# --------------------------------------------------------------------------
interval_signal_file <- file.path(DOMAIN_DIR, "interval_cenpa_signal.csv")
interval_signal <- fread(interval_signal_file)
interval_signal[, interval_key := paste(chrom, start, end, sep = "_")]

gap_dt <- merge(centro[, .(chrom, start, end, interval_key, gap_class)],
                interval_signal[, .(interval_key, fpkm_150, fpkm_151)],
                by = "interval_key")

gap_dt <- merge(gap_dt, chr_bg, by = "chrom")

# log2 fold change over chromosome-mean background
gap_dt[, cenpa_150_norm := log2((fpkm_150 + PSEUDO) / (chr_mean_150 + PSEUDO))]
gap_dt[, cenpa_151_norm := log2((fpkm_151 + PSEUDO) / (chr_mean_151 + PSEUDO))]
gap_dt[, cenpa_norm := (cenpa_150_norm + cenpa_151_norm) / 2]

message("Merged: ", nrow(gap_dt), " intervals")

# --------------------------------------------------------------------------
# Step 4: Summary
# --------------------------------------------------------------------------
chr_delta <- gap_dt[, .(
  med_tight = median(cenpa_norm[gap_class == "tight"], na.rm = TRUE),
  med_isolated = median(cenpa_norm[gap_class == "isolated"], na.rm = TRUE)
), by = chrom]
chr_delta <- chr_delta[!is.na(med_tight) & !is.na(med_isolated)]
chr_delta[, delta := med_tight - med_isolated]

obs_delta <- median(gap_dt$cenpa_norm[gap_dt$gap_class == "tight"], na.rm = TRUE) -
             median(gap_dt$cenpa_norm[gap_dt$gap_class == "isolated"], na.rm = TRUE)

cat(sprintf("
Overall median delta (tight - isolated): %.3f
", obs_delta))
cat(sprintf("Chr with tight < isolated: %d / %d
", sum(chr_delta$delta < 0), nrow(chr_delta)))

# --------------------------------------------------------------------------
# Figure: Boxplot + jitter, tight LEFT, isolated RIGHT
# --------------------------------------------------------------------------
gap_dt[, gap_class := factor(gap_class, levels = c("tight", "isolated"))]

p_box <- ggplot(gap_dt, aes(x = gap_class, y = cenpa_norm, fill = gap_class)) +
  geom_hline(yintercept = 0, linetype = "dashed", linewidth = 0.5, color = "grey50") +
  geom_boxplot(outlier.shape = NA, linewidth = 0.6, alpha = 0.85) +
  geom_jitter(size = 0.3, alpha = 0.25, width = 0.2, color = "grey30") +
  scale_fill_manual(values = c("tight" = "#2166AC", "isolated" = "#D6604D"), guide = "none") +
  scale_x_discrete(labels = c(
    "tight" = sprintf("Tight
(< 700 bp gap)
n = %d", n_tight),
    "isolated" = sprintf("Isolated
(≥ 700 bp gap)
n = %d", n_isolated)
  )) +
  labs(
    x = NULL,
    y = expression("log"[2] * "(CENP-A FPKM / chromosome-mean)"),
    title = "CENP-A depleted in tight HOR clusters",
    subtitle = sprintf(
      "Normalized to genome-wide chr-mean background  |  Median delta = %.2f  |  %d/%d chr tight < isolated",
      obs_delta, sum(chr_delta$delta < 0), nrow(chr_delta))
  ) +
  coord_cartesian(ylim = c(-8, 8)) +
  theme_cenpa +
  theme(panel.grid.minor = element_blank())

print(p_box)

# --------------------------------------------------------------------------
# Save
# --------------------------------------------------------------------------
ggsave(
  filename = file.path("supp_gap_class_enrichment.pdf"),
  plot = p_box,
  width = 5, height = 5, units = "in",
  device = cairo_pdf
)

ggsave(
  filename = file.path("supp_gap_class_enrichment.png"),
  plot = p_box,
  width = 5, height = 5, units = "in",
  dpi = 600, bg = "white"
)

cat("
Saved: supp_gap_class_enrichment.pdf / .png
")



---
## Karyotype: CENP-A Signal and centroAnno/HiCAT Annotations

Genome-wide karyotype showing CENP-A CUT&Tag signal (25-kb bins) alongside
centroAnno-predicted satellite arrays and CENP-A enriched satellite arrays.
chr4 and chr25 also show HiCAT HOR array tracks.


In [ ]:
# ============================================================================
# Genome-wide CENP-A + centroAnno satellite arrays (repeat regions)
# Now runs from standalone R script to keep notebook lean.
# ============================================================================

source("karyotype_centroAnno_satellite.R")

# Display the output PNG
if (requireNamespace("IRdisplay", quietly = TRUE)) {
  IRdisplay::display_png(file = file.path(
    "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj",
    "figure/cenpa-cuttag-enrichment",
    "genome_wide_CENPA_centroAnno_satellite.png"
  ))
}


In [ ]:
# ============================================================================
# Genome-wide CENP-A + centroAnno HOR arrays
# Standalone R script — centroAnno high-confidence HOR predictions.
# ============================================================================

source("karyotype_centroAnno_HORs.R")

# Display the output PNG
if (requireNamespace("IRdisplay", quietly = TRUE)) {
  IRdisplay::display_png(file = file.path(
    "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj",
    "figure/cenpa-cuttag-enrichment",
    "genome_wide_CENPA_centroAnno_HORs.png"
  ))
}


In [ ]:
# CENP-A signal + HiCAT HOR arrays — chr4 only
# Sources the external R script: hicat_karyotype_chr4.R

HICAT_DIR <- "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/figure/cenpa-cuttag-enrichment"
source(file.path(HICAT_DIR, "hicat_karyotype_chr4.R"))


In [ ]:
# ============================================================================
# Replicate concordance
# ============================================================================

# CENP-A replicate correlation at domain level
rep_domain <- dcast(domain_counts[sample %in% c("XG_150", "XG_151")],
                    chrom + start + end + domain_id ~ sample, value.var = "log2_signal")
cor_domain <- cor(rep_domain$XG_150, rep_domain$XG_151,
                  method = "spearman", use = "complete.obs")
message("Domain-level Spearman rho (XG_150 vs XG_151): ", round(cor_domain, 3))

# H3K27ac replicate correlation
rep_h3k27ac <- dcast(domain_counts[sample %in% c("XG_152", "XG_153")],
                      chrom + start + end + domain_id ~ sample, value.var = "log2_signal")
if (all(c("XG_152", "XG_153") %in% names(rep_h3k27ac))) {
  cor_h3k27ac <- cor(rep_h3k27ac$XG_152, rep_h3k27ac$XG_153,
                      method = "spearman", use = "complete.obs")
  message("H3K27ac replicate Spearman rho (XG_152 vs XG_153): ", round(cor_h3k27ac, 3))
}

# Quick scatter of CENP-A replicates
p_rep <- ggplot(rep_domain, aes(x = XG_150, y = XG_151)) +
  geom_point(alpha = 0.4, size = 0.8, color = "#2166AC") +
  geom_smooth(method = "lm", se = TRUE, color = "grey30", linewidth = 0.5) +
  annotate("text", x = min(rep_domain$XG_150, na.rm = TRUE),
           y = max(rep_domain$XG_151, na.rm = TRUE),
           label = paste0("Spearman rho = ", round(cor_domain, 3)),
           hjust = 0, vjust = 1, size = 3.5) +
  labs(x = expression("CENP-A rep1 log"[2] * "(normalized signal)"),
       y = expression("CENP-A rep2 log"[2] * "(normalized signal)"),
       title = "CENP-A replicate concordance at merged domains") +
  theme_cenpa
print(p_rep)


---
## Chromosome-level CUT&Tag Signal Enrichment

**Background model:** Chromosome- and length-matched shuffle iterations. For each chromosome, the observed statistic is compared against the full shuffled distribution to derive an empirical P value.

**Definition:**
- **Observed** (O_c): median log₂(normalized CUT&Tag signal) across domains on chromosome c
- **Expected** (B_{c,i}): median log₂(normalized CUT&Tag signal) across shuffled intervals for iteration i on chromosome c
- **Enrichment** (Δ_c): O_c − median_i(B_{c,i}) — the chromosome-level CUT&Tag signal relative to the median randomized expectation
- **Fold enrichment**: 2^{Δ_c}
- **Empirical P**: (1 + count of shuffles where B_{c,i} ≥ O_c) / (N_iterations + 1), with Benjamini-Hochberg correction across 30 chromosomes

All 30 chromosomes show positive Δ_c. Effect sizes and FDR-corrected empirical P values are reported below.</text>

In [ ]:
# ============================================================================
# Chromosome-level enrichment with empirical P values (all shuffle iterations)
# ============================================================================

# Auto-detect number of iterations from data
N_SHUFFLES <- uniqueN(bg1$iter_id)
message("Using ", N_SHUFFLES, " shuffle iterations")

# ---- Per-chromosome OBSERVED: median log2_signal at domains ----
chr_obs <- domain_counts[, .(
  O_c = median(log2_signal, na.rm = TRUE),
  n_domains = .N
), by = .(sample, chrom)]

# ---- Per-chromosome SHUFFLED: median log2_signal per iteration ----
# bg1 has columns: chrom, start, end, iter_id, count, sample, length, norm_signal, log2_signal
chr_bg <- bg1[, .(
  B_ci = median(log2_signal, na.rm = TRUE)
), by = .(sample, chrom, iter_id)]

# ---- Median background across all iterations ----
chr_bg_median <- chr_bg[, .(
  median_bg = median(B_ci, na.rm = TRUE),
  q25_bg = quantile(B_ci, 0.25, na.rm = TRUE),
  q75_bg = quantile(B_ci, 0.75, na.rm = TRUE)
), by = .(sample, chrom)]

# ---- Merge observed with background median ----
chr_result <- merge(chr_obs, chr_bg_median, by = c("sample", "chrom"))
chr_result[, delta := O_c - median_bg]
chr_result[, fold_enrichment := 2^delta]
chr_result[, sample_label := factor(SAMPLE_LABELS[sample],
                                     levels = unname(SAMPLE_LABELS[SAMPLES]))]

# ---- Empirical P value: compare observed vs each iteration ----
# For each chromosome, count how many shuffle iterations have B_ci >= O_c
chr_pvals <- merge(chr_bg, chr_obs[, .(sample, chrom, O_c)],
                   by = c("sample", "chrom"))
chr_pvals[, exceeds := B_ci >= O_c]
chr_empirical_p <- chr_pvals[, .(
  n_exceed = sum(exceeds, na.rm = TRUE),
  n_total = .N
), by = .(sample, chrom)]
chr_empirical_p[, p_empirical := (n_exceed + 1) / (n_total + 1)]

# Merge P values into results
chr_result <- merge(chr_result, chr_empirical_p[, .(sample, chrom, p_empirical)],
                    by = c("sample", "chrom"))

# ---- Benjamini-Hochberg correction (per sample, across 30 chromosomes) ----
chr_result[, p_adj := p.adjust(p_empirical, method = "BH"), by = sample]
chr_result[, fdr_significant := p_adj < 0.05]

# ---- Order chromosomes by mean CENP-A delta ----
chr_order_data <- chr_result[sample %in% c("XG_150", "XG_151"),
                              .(mean_delta = mean(delta, na.rm = TRUE)), by = chrom]
setorder(chr_order_data, mean_delta)
chr_result[, chrom_ordered := factor(chrom, levels = chr_order_data$chrom)]

# ---- Print detailed per-chromosome results for CENP-A rep1 ----
cat("\n=== Per-chromosome enrichment (CENP-A rep1, XG_150) ===\n")
cat(sprintf("Background: %d chromosome-matched shuffle iterations\n", N_SHUFFLES))
cat(sprintf("%-8s %8s %8s %8s %8s %8s %8s %s\n",
            "Chrom", "O_c", "Bg_med", "Delta", "FoldEnr", "P_emp", "P_BH", "FDR<0.05"))
cat(strrep("-", 76), "\n")
chr_print <- chr_result[sample == "XG_150"][order(-delta)]
for (r in seq_len(nrow(chr_print))) {
  cat(sprintf("%-8s %8.3f %8.3f %8.3f %8.2f %8.4f %8.4f %s\n",
              chr_print$chrom[r],
              chr_print$O_c[r],
              chr_print$median_bg[r],
              chr_print$delta[r],
              chr_print$fold_enrichment[r],
              chr_print$p_empirical[r],
              chr_print$p_adj[r],
              ifelse(chr_print$fdr_significant[r], "*", "")))
}

# ---- Summary statistics ----
cat("\n=== Summary across chromosomes ===\n")
for (s in c("XG_150", "XG_151")) {
  d <- chr_result[sample == s]
  cat(sprintf("\n%s (%s):\n", s, SAMPLE_LABELS[s]))
  cat(sprintf("  Chromosomes with positive delta: %d/%d\n",
              sum(d$delta > 0), nrow(d)))
  cat(sprintf("  Delta range: %.2f to %.2f (median: %.2f)\n",
              min(d$delta), max(d$delta), median(d$delta)))
  cat(sprintf("  Fold enrichment range: %.2fx to %.2fx (median: %.2fx)\n",
              min(d$fold_enrichment), max(d$fold_enrichment),
              median(d$fold_enrichment)))
  cat(sprintf("  FDR < 0.05: %d/%d chromosomes\n",
              sum(d$fdr_significant), nrow(d)))
  cat(sprintf("  P_empirical range: %.4f to %.4f\n",
              min(d$p_empirical), max(d$p_empirical)))
}

# ---- Classification by tertile (magnitude of enrichment) ----
chr_delta_wide <- dcast(chr_result[sample %in% c("XG_150", "XG_151")],
                        chrom ~ sample, value.var = "delta")
chr_delta_wide[, mean_delta := (XG_150 + XG_151) / 2]

delta_tertiles <- quantile(chr_delta_wide$mean_delta,
                           probs = c(0, 1/3, 2/3, 1), na.rm = TRUE)
chr_classification <- chr_delta_wide[, .(
  chrom,
  XG_150_delta = XG_150,
  XG_151_delta = XG_151,
  mean_delta,
  category = fcase(
    mean_delta >= delta_tertiles[3], "Strong enrichment",
    mean_delta >= delta_tertiles[2], "Moderate enrichment",
    mean_delta >= delta_tertiles[1], "Weak enrichment",
    default = "Insufficient data"
  )
)]

cat("\nChromosome classification (tertile-based):\n")
print(table(chr_classification$category))

# ---- Beeswarm + boxplot ----
chr_delta_plot <- chr_result[sample %in% c("XG_150", "XG_151")]
chr_delta_plot <- merge(chr_delta_plot,
                        chr_classification[, .(chrom, category)],
                        by = "chrom", all.x = TRUE)

# Annotation text for FDR counts
n_fdr_150 <- chr_result[sample == "XG_150", sum(fdr_significant)]
n_fdr_151 <- chr_result[sample == "XG_151", sum(fdr_significant)]

p_chrom_enrich <- ggplot(chr_delta_plot, aes(x = "All chromosomes", y = delta)) +
  geom_hline(yintercept = 0, linetype = "dashed", color = "grey50", linewidth = 0.3) +
  geom_jitter(aes(color = category), size = 2.5, width = 0.2, alpha = 0.8) +
  geom_boxplot(fill = NA, outlier.shape = NA, width = 0.3, linewidth = 0.4) +
  facet_wrap(~ sample_label, nrow = 1) +
  scale_color_manual(values = cat_colors, name = "Enrichment\nmagnitude") +
  labs(x = NULL,
       y = expression(Delta * " log"[2] * "(normalized CUT&Tag signal) [domain − shuffle median]"),
       title = "Chromosome-level CENP-A enrichment at CENP-A enriched satellite arrays",
       subtitle = paste0(
         N_SHUFFLES, " chromosome-matched shuffle iterations; empirical P with BH correction\n",
         "All 30/30 chromosomes enriched (",
         "FDR < 0.05: ", n_fdr_150, " (rep1), ", n_fdr_151, " (rep2) chromosomes); ",
         "Δ range: ", round(min(chr_classification$mean_delta), 2),
         " – ", round(max(chr_classification$mean_delta), 2))) +
  theme_cenpa +
  theme(axis.text.x = element_blank(), axis.ticks.x = element_blank())

print(p_chrom_enrich)

In [ ]:
# ============================================================================
# Publication-ready per-chromosome CENP-A enrichment bar plot
# ============================================================================

library(ggplot2)
library(scales)

p_chrom_bar <- ggplot(
  chr_delta_plot,
  aes(
    x = chrom_ordered,
    y = delta,
    fill = sample
  )
) +

  # Zero-reference line
  geom_hline(
    yintercept = 0,
    linewidth = 0.55,
    colour = "black"
  ) +

  # Replicate bars
  geom_col(
    position = position_dodge2(
      width = 0.78,
      preserve = "single",
      padding = 0.08
    ),
    width = 0.72,
    alpha = 1
  ) +

  # Replicate colors and labels
  scale_fill_manual(
    values = SAMPLE_COLORS,
    labels = SAMPLE_LABELS,
    guide = guide_legend(
      nrow = 1,
      byrow = TRUE,
      override.aes = list(alpha = 1)
    )
  ) +

  # Clean y-axis scaling
  scale_y_continuous(
    breaks = breaks_pretty(n = 5),
    expand = expansion(mult = c(0, 0.06))
  ) +

  # More concise and precise labeling
  labs(
    x = "Chromosome",
    y = expression(
      Delta ~ "median log"[2] * "(normalized CUT&Tag signal)"
    ),
    fill = NULL,
    title = "CENP-A enrichment by chromosome",
    subtitle = "Chromosomes ordered by mean enrichment across replicates"
  ) +

  theme_cenpa +

  theme(
    # Titles
    plot.title = element_text(
      size = 12,
      face = "bold",
      hjust = 0,
      margin = margin(b = 3)
    ),
    plot.subtitle = element_text(
      size = 9,
      colour = "grey30",
      hjust = 0,
      margin = margin(b = 10)
    ),

    # Axis titles
    axis.title.x = element_text(
      size = 10,
      margin = margin(t = 9)
    ),
    axis.title.y = element_text(
      size = 10,
      margin = margin(r = 9)
    ),

    # Axis tick labels
    axis.text.x = element_text(
      size = 7.5,
      angle = 50,
      hjust = 1,
      vjust = 1
    ),
    axis.text.y = element_text(
      size = 8.5,
      colour = "black"
    ),

    # Remove clutter from categorical vertical grid lines
    panel.grid.major.x = element_blank(),
    panel.grid.minor = element_blank(),
    panel.grid.major.y = element_line(
      linewidth = 0.35,
      colour = "grey88"
    ),

    # Keep a clean panel boundary
    panel.border = element_rect(
      colour = "grey35",
      fill = NA,
      linewidth = 0.55
    ),

    # Compact horizontal legend
    legend.position = "top",
    legend.justification = "left",
    legend.direction = "horizontal",
    legend.text = element_text(size = 9),
    legend.key.width = unit(0.75, "cm"),
    legend.key.height = unit(0.38, "cm"),
    legend.margin = margin(b = 4),
    legend.box.margin = margin(0, 0, 0, 0),

    # Figure margins
    plot.margin = margin(
      t = 8,
      r = 10,
      b = 6,
      l = 8
    )
  )

print(p_chrom_bar)

# Recommended landscape export
ggsave(
  filename = file.path( "CENPA_per_chromosome_enrichment.pdf"),
  plot = p_chrom_bar,
  width = 9.0,
  height = 5.2,
  units = "in",
  device = cairo_pdf
)

ggsave(
  filename = file.path("CENPA_per_chromosome_enrichment.png"),
  plot = p_chrom_bar,
  width = 9.0,
  height = 5.2,
  units = "in",
  dpi = 600,
  bg = "white"
)

---
## Null Distribution: Observed vs Shuffled Background (per chromosome)

For a single chromosome, the observed median domain signal (vertical line) is overlaid on the density of all shuffle-iteration medians (the null distribution). The further the observed value lies from the null mode, the stronger the enrichment evidence. The empirical P value is the fraction of shuffle iterations with a median ≥ the observed value.

In [ ]:
# ============================================================================
# Publication-ready null distribution
# chr1, CENP-A rep1 only
# ============================================================================

library(data.table)
library(ggplot2)
library(scales)

# ----------------------------------------------------------------------------
# Select chromosome and sample
# ----------------------------------------------------------------------------

DENSITY_CHROM <- "chr1"
TARGET_LABEL  <- "CENP-A rep1"

# Identify the internal sample name associated with the display label
TARGET_SAMPLE <- names(SAMPLE_LABELS)[
  unname(SAMPLE_LABELS) == TARGET_LABEL
]

if (length(TARGET_SAMPLE) != 1L) {
  stop(
    "Could not uniquely identify the sample corresponding to '",
    TARGET_LABEL,
    "'. Check SAMPLE_LABELS."
  )
}

sample_color <- unname(SAMPLE_COLORS[TARGET_SAMPLE])

if (length(sample_color) != 1L || is.na(sample_color)) {
  stop("No color was found for the selected sample in SAMPLE_COLORS.")
}

# ----------------------------------------------------------------------------
# Extract shuffled and observed values
# ----------------------------------------------------------------------------

null_values <- chr_bg[
  chrom == DENSITY_CHROM & sample == TARGET_SAMPLE,
  B_ci
]

null_values <- null_values[is.finite(null_values)]

obs_row <- chr_obs[
  chrom == DENSITY_CHROM & sample == TARGET_SAMPLE
]

pval_row <- chr_empirical_p[
  chrom == DENSITY_CHROM & sample == TARGET_SAMPLE
]

if (length(null_values) < 2L) {
  stop("Fewer than two finite shuffled values were found.")
}

if (nrow(obs_row) != 1L) {
  stop(
    "Expected exactly one observed value for ",
    DENSITY_CHROM, ", ", TARGET_LABEL,
    "; found ", nrow(obs_row), "."
  )
}

observed_value <- obs_row$O_c[[1]]
empirical_p    <- pval_row$p_empirical[[1]]

if (!is.finite(observed_value)) {
  stop("The observed value is not finite.")
}

# ----------------------------------------------------------------------------
# Compute density and summary statistics
# ----------------------------------------------------------------------------

density_object <- density(
  null_values,
  n = 1024,
  from = min(null_values) - 0.35,
  to   = max(null_values) + 0.35
)

density_dt <- data.table(
  x = density_object$x,
  density = density_object$y
)

null_median <- median(null_values, na.rm = TRUE)
n_shuffles  <- length(null_values)
max_density <- max(density_dt$density, na.rm = TRUE)

# Include the observed value comfortably inside the x-axis range
all_x <- c(density_dt$x, null_median, observed_value)
x_span <- diff(range(all_x))

if (!is.finite(x_span) || x_span == 0) {
  x_span <- 1
}

x_limits <- c(
  min(all_x) - 0.04 * x_span,
  max(all_x) + 0.08 * x_span
)

# Place the observed label on the side facing into the panel
observed_hjust <- if (
  observed_value > mean(x_limits)
) 1.08 else -0.08

# Empirical P-value label
p_label <- if (is.na(empirical_p)) {
  "italic(P)[emp] == NA"
} else {
  sprintf("italic(P)[emp] == %.4f", empirical_p)
}

# ----------------------------------------------------------------------------
# Build plot
# ----------------------------------------------------------------------------

p_null_density_chr1 <- ggplot(
  density_dt,
  aes(x = x, y = density)
) +

  # Shuffled-background distribution
  geom_area(
    fill = "grey82",
    alpha = 0.90
  ) +
  geom_line(
    colour = "grey35",
    linewidth = 0.70
  ) +

  # Median across the shuffle-iteration medians
  geom_vline(
    xintercept = null_median,
    colour = "grey30",
    linewidth = 0.75,
    linetype = "dashed"
  ) +

  # Observed centromeric-domain median
  geom_vline(
    xintercept = observed_value,
    colour = sample_color,
    linewidth = 1.25
  ) +

  # Empirical P value
  annotate(
    "text",
    x = x_limits[1] + 0.03 * diff(x_limits),
    y = max_density * 0.95,
    label = p_label,
    parse = TRUE,
    hjust = 0,
    vjust = 0,
    size = 4.4
  ) +

  # Label the median shuffled signal
  annotate(
    "text",
    x = null_median,
    y = max_density * 0.72,
    label = sprintf(
      "Median shuffled\nsignal = %.2f",
      null_median
    ),
    colour = "grey25",
    hjust = 1.08,
    vjust = 0.5,
    size = 3.6
  ) +

  # Label the observed signal
  annotate(
    "text",
    x = observed_value,
    y = max_density * 0.92,
    label = sprintf(
      "Observed = %.2f",
      observed_value
    ),
    colour = sample_color,
    fontface = "bold",
    hjust = observed_hjust,
    vjust = 0,
    size = 4.4
  ) +

  scale_x_continuous(
    limits = x_limits,
    breaks = breaks_pretty(n = 6),
    expand = expansion(mult = 0)
  ) +

  scale_y_continuous(
    breaks = breaks_pretty(n = 4),
    expand = expansion(mult = c(0, 0.05))
  ) +

  labs(
    x = expression(
      "Median log"[2] * "(normalized CUT&Tag signal)"
    ),
    y = "Probability density",
    title = paste0(
        DENSITY_CHROM,
        " CENP-A enrichment relative to shuffled background"
    ),
    subtitle = paste0(
      TARGET_LABEL, "; ",
      format(n_shuffles, big.mark = ","),
      " random shuffle iterations of CENP-A enriched satellite arrays\n",
      "matched by chromosome and interval length"
    )
  ) +

  theme_cenpa +

  theme(
    plot.title = element_text(
      size = 13,
      face = "bold",
      hjust = 0,
      margin = margin(b = 3)
    ),
    plot.subtitle = element_text(
      size = 10,
      colour = "grey30",
      hjust = 0,
      lineheight = 1.15,
      margin = margin(b = 10)
    ),
    axis.title.x = element_text(
      size = 11,
      margin = margin(t = 9)
    ),
    axis.title.y = element_text(
      size = 11,
      margin = margin(r = 9)
    ),
    axis.text = element_text(
      size = 9.5,
      colour = "black"
    ),
    panel.grid.major.x = element_blank(),
    panel.grid.minor = element_blank(),
    panel.grid.major.y = element_line(
      colour = "grey88",
      linewidth = 0.35
    ),
    panel.border = element_rect(
      colour = "grey35",
      fill = NA,
      linewidth = 0.55
    ),
    plot.margin = margin(
      t = 8,
      r = 14,
      b = 8,
      l = 8
    )
  )

print(p_null_density_chr1)

# ----------------------------------------------------------------------------
# Save
# ----------------------------------------------------------------------------

ggsave(
  filename = file.path(
    "CENPA_rep1_chr1_null_distribution.pdf"
  ),
  plot = p_null_density_chr1,
  width = 5.6,
  height = 4.8,
  units = "in",
  device = cairo_pdf
)

ggsave(
  filename = file.path(
    "CENPA_rep1_chr1_null_distribution.png"
  ),
  plot = p_null_density_chr1,
  width = 5.6,
  height = 4.8,
  units = "in",
  dpi = 600,
  bg = "white"
)

cat(
  sprintf(
    "\nShowing %s for %s using %s shuffle iterations.\n",
    TARGET_LABEL,
    DENSITY_CHROM,
    format(n_shuffles, big.mark = ",")
  )
)

---
## Signal Concentration: Top-Window to Chromosome-Mean Ratio

**Motivation:** The whole-chromosome Gini is confounded by genome-wide signal
sparsity. We instead ask: how many times higher is the dominant 25-kb window
than the average (mean) window on that chromosome?

**Why mean, not median?** On large chromosomes the median 25-kb window has
zero CENP-A signal (the centromere spans <1% of the chromosome). Dividing by
near-zero medians produces uninterpretable ratios (>20,000×). The mean is
always positive and the ratio `top / mean` is numerically stable:
it equals `top1_fraction × n_bins`.

**Interpretation:**
- **Ratio ≈ 1**: no peak — all windows have similar signal
- **Ratio ≈ n_bins** (e.g., ~6000 for chr4): all signal is in one window
- **CENP-A should show high ratios** (focal centromere); **H3K27ac low**
  (distributed across many regulatory elements)


In [ ]:
# ============================================================================
# Per-chromosome top-window to chromosome-mean ratio
#
# For each chromosome, 25-kb binned BigWig signal → ratio of the single
# highest window to the chromosome-wide mean. High ratio = one dominant
# peak towering over background; low ratio = distributed signal.
#
# Using mean (not median) avoids division-by-near-zero when >50% of windows
# have zero signal. top/mean = top1_fraction × n_bins.
# ============================================================================

suppressPackageStartupMessages({
  library(rtracklayer)
  library(GenomicRanges)
})

BIN_WIDTH <- 25000L

# --------------------------------------------------------------------------
# BigWig file paths (all 4 samples)
# --------------------------------------------------------------------------
BW_DIR <- file.path(
  "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj",
  "figure/cenpa-cuttag-centromere/bw_files"
)

peak_bw_files <- list(
  "XG_150" = file.path(BW_DIR, "XG_150.all.bw"),
  "XG_151" = file.path(BW_DIR, "XG_151.all.bw"),
  "XG_152" = file.path(BW_DIR, "XG_152.all.bw"),
  "XG_153" = file.path(BW_DIR, "XG_153.all.bw")
)

# --------------------------------------------------------------------------
# Chromosome sizes
# --------------------------------------------------------------------------
peak_fai <- file.path(
  "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj",
  "degu-genome-browser-pythonVersion",
  "assembly_final.sorted.headerRenamed.chrAssigned.mito.fasta.fai"
)

peak_chr_sizes <- fread(peak_fai, header = FALSE, select = c(1, 2),
                        col.names = c("chrom", "size"))
peak_chr_sizes <- peak_chr_sizes[chrom %in% CHROMOSOMES]
peak_chr_lengths <- setNames(peak_chr_sizes$size, peak_chr_sizes$chrom)

# --------------------------------------------------------------------------
# Compute top / mean ratio for each chromosome × sample
# --------------------------------------------------------------------------
peak_results <- list()

for (sample_name in names(peak_bw_files)) {

  bw_file <- peak_bw_files[[sample_name]]
  message("Processing ", sample_name, " (", SAMPLE_LABELS[sample_name], ") ...")

  bw <- BigWigFile(bw_file)

  for (chrom_i in CHROMOSOMES) {

    chr_len <- peak_chr_lengths[[chrom_i]]
    n_bins <- max(1L, as.integer(ceiling(chr_len / BIN_WIDTH)))

    summarized <- summary(
      bw,
      size = n_bins,
      type = "mean",
      which = GRanges(
        seqnames = chrom_i,
        ranges = IRanges(start = 1, end = chr_len)
      )
    )[[1]]

    scores <- as.numeric(summarized$score)
    scores[is.na(scores) | !is.finite(scores)] <- 0
    scores <- pmax(scores, 0)

    total     <- sum(scores)
    top_val   <- max(scores)
    mean_val  <- total / length(scores)
    ratio <- if (mean_val > 0) top_val / mean_val else NA_real_

    # Sanity: ratio should equal top1_fraction * n_bins
    top1_frac <- if (total > 0) top_val / total else NA_real_

    peak_results[[paste(sample_name, chrom_i, sep = "_")]] <- data.table(
      sample = sample_name,
      chrom = chrom_i,
      chr_length_bp = chr_len,
      n_bins = length(scores),
      total_signal = total,
      top_window = top_val,
      mean_window = mean_val,
      top_mean_ratio = ratio,
      top1_fraction = top1_frac
    )
  }
}

peak_dt <- rbindlist(peak_results)

# Add display labels
peak_dt[, antibody := ifelse(sample %in% c("XG_150", "XG_151"),
                             "CENP-A", "H3K27ac")]
peak_dt[, replicate := ifelse(sample %in% c("XG_150", "XG_152"),
                               "rep1", "rep2")]
peak_dt[, sample_label := SAMPLE_LABELS[sample]]

message("Done. ", nrow(peak_dt), " chromosome × sample combinations.")

# --------------------------------------------------------------------------
# Order chromosomes by mean CENP-A ratio (descending)
# --------------------------------------------------------------------------
peak_cenpa_mean <- peak_dt[antibody == "CENP-A",
                          .(mean_ratio = mean(top_mean_ratio, na.rm = TRUE)),
                          by = chrom]
setorder(peak_cenpa_mean, -mean_ratio)
peak_dt[, chrom := factor(chrom, levels = peak_cenpa_mean$chrom)]

# --------------------------------------------------------------------------
# Print summary
# --------------------------------------------------------------------------
cat("\n")
cat("=== Top-window / chromosome-mean ratio ===\n")
cat(sprintf("CENP-A median ratio: %.0f (rep1), %.0f (rep2)\n",
            median(peak_dt[sample == "XG_150", top_mean_ratio], na.rm = TRUE),
            median(peak_dt[sample == "XG_151", top_mean_ratio], na.rm = TRUE)))
cat(sprintf("H3K27ac median ratio: %.0f (rep1), %.0f (rep2)\n",
            median(peak_dt[sample == "XG_152", top_mean_ratio], na.rm = TRUE),
            median(peak_dt[sample == "XG_153", top_mean_ratio], na.rm = TRUE)))

# Top and bottom chromosomes
cat("\nTop 5 chromosomes (CENP-A rep1):\n")
top5 <- peak_dt[sample == "XG_150"][order(-top_mean_ratio)][1:5]
print(top5[, .(chrom, top_mean_ratio, top_window, mean_window, n_bins, top1_fraction)])

cat("\nBottom 5 chromosomes (CENP-A rep1):\n")
bot5 <- peak_dt[sample == "XG_150"][order(top_mean_ratio)][1:5]
print(bot5[, .(chrom, top_mean_ratio, top_window, mean_window, n_bins, top1_fraction)])

# Replicate concordance
peak_wide <- dcast(peak_dt, chrom ~ sample, value.var = "top_mean_ratio")
cor_cenpa <- cor(peak_wide$XG_150, peak_wide$XG_151,
                method = "spearman", use = "complete.obs")
cor_h3k27ac <- cor(peak_wide$XG_152, peak_wide$XG_153,
                   method = "spearman", use = "complete.obs")
cat(sprintf("\nCENP-A replicate Spearman rho: %.3f\n", cor_cenpa))
cat(sprintf("H3K27ac replicate Spearman rho: %.3f\n", cor_h3k27ac))

# --------------------------------------------------------------------------
# Plot: grouped barplot, all 4 samples
# --------------------------------------------------------------------------

p_peak_ratio <- ggplot(
  peak_dt,
  aes(x = chrom, y = top_mean_ratio, fill = sample_label)
) +
  geom_col(
    position = position_dodge(width = 0.8),
    width = 0.7,
    color = "white",
    linewidth = 0.15
  ) +
  scale_fill_manual(
    values = c(
      "CENP-A rep1" = "#2166AC",
      "CENP-A rep2" = "#92C5DE",
      "H3K27ac" = "#B2182B",
      "H3K27ac rep2" = "#D6604D"
    ),
    name = NULL
  ) +
  geom_hline(
    yintercept = 1,
    linewidth = 0.4,
    linetype = "dashed",
    color = "grey50"
  ) +
  scale_y_continuous(
    expand = expansion(mult = c(0, 0.05))
  ) +
  labs(
    x = NULL,
    y = "Top 25-kb window / chromosome mean",
    title = "CENP-A and H3K27ac peak dominance per chromosome",
    subtitle = paste0(
      "Ratio of highest 25-kb window to chromosome-wide mean. ",
      "Dashed line at 1 = no peak.\n",
      "CENP-A rep Spearman ρ = ", round(cor_cenpa, 3),
      "; H3K27ac rep Spearman ρ = ", round(cor_h3k27ac, 3)
    )
  ) +
  theme_cenpa +
  theme(
    axis.text.x = element_text(
      angle = 45,
      hjust = 1,
      vjust = 1,
      size = 8
    ),
    legend.position = "bottom",
    legend.margin = margin(t = 4),
    panel.grid.major.x = element_blank(),
    plot.subtitle = element_text(lineheight = 1.15)
  )

print(p_peak_ratio)

# --------------------------------------------------------------------------
# Save
# --------------------------------------------------------------------------
ggsave(
  filename = file.path("supp_top_window_to_mean_ratio.pdf"),
  plot = p_peak_ratio,
  width = 14,
  height = 5,
  units = "in",
  device = cairo_pdf
)

ggsave(
  filename = file.path("supp_top_window_to_mean_ratio.png"),
  plot = p_peak_ratio,
  width = 9,
  height = 5,
  units = "in",
  dpi = 600,
  bg = "white"
)

ggsave(
  filename = file.path("supp_top_window_to_mean_ratio.svg"),
  plot = p_peak_ratio,
  width =9,
  height = 5,
  units = "in",
  dpi = 600,
  bg = "white"
)

cat("\nSaved: supp_top_window_to_mean_ratio.pdf / .png\n")


---
## Array-centered Metaprofile

**What is a domain?** Domains are merged centroAnno intervals — adjacent centromeric repeat predictions within 250 kb are merged into contiguous "domains," each representing a putative centromere. There are 492 domains across 30 chromosomes (1–84 per chromosome, median size ~60 kb).

CENP-A and H3K27ac normalized signal is aggregated across all domains, centered on domain boundaries. The grey rectangle marks the domain interior (core + margins). Flanks extend ±1 Mb.</text>

In [ ]:
# ============================================================================
# Publication-ready array-centered metaprofile (+/- 1 Mb flanks)
# ============================================================================

library(ggplot2)
library(scales)
library(grid)

# ----------------------------------------------------------------------------
# Define bin order
# ----------------------------------------------------------------------------
bin_order_domain <- c(
  "left_500000_1000000", "left_100000_500000", "left_10000_100000",
  "left_1000_10000", "left_0_1000",
  "left_margin_0_50kb", "domain_core", "right_margin_0_50kb",
  "right_0_1000", "right_1000_10000", "right_10000_100000",
  "right_100000_500000", "right_500000_1000000"
)

# Display labels
bin_labels_display <- c(
  "left_500000_1000000" = "-1 Mb to -500 kb",
  "left_100000_500000"  = "-500 kb to -100 kb",
  "left_10000_100000"   = "-100 kb to -10 kb",
  "left_1000_10000"     = "-10 kb to -1 kb",
  "left_0_1000"         = "-1 kb to 0",
  "left_margin_0_50kb"  = "Domain\n(left)",
  "domain_core"         = "Domain\n(core)",
  "right_margin_0_50kb" = "Domain\n(right)",
  "right_0_1000"        = "0 to +1 kb",
  "right_1000_10000"    = "+1 kb to +10 kb",
  "right_10000_100000"  = "+10 kb to +100 kb",
  "right_100000_500000" = "+100 kb to +500 kb",
  "right_500000_1000000"= "+500 kb to +1 Mb"
)

# ----------------------------------------------------------------------------
# Keep only bins present in the data
# ----------------------------------------------------------------------------
existing_bins <- intersect(bin_order_domain, unique(domain_dist$bin_label))
existing_labels <- bin_labels_display[existing_bins]

# ----------------------------------------------------------------------------
# Aggregate across domains
# ----------------------------------------------------------------------------
domain_meta <- domain_dist[, .(
  mean_signal = mean(log2_signal, na.rm = TRUE),
  se_signal   = sd(log2_signal, na.rm = TRUE) / sqrt(.N),
  n_domains   = uniqueN(domain_id)
), by = .(sample, bin_label)]

domain_meta[, bin_factor := factor(
  bin_label,
  levels = existing_bins,
  labels = existing_labels
)]

domain_meta[, sample_label := SAMPLE_LABELS[sample]]

n_domains_total <- uniqueN(domain_dist$domain_id)

# Domain interior span
domain_start <- which(existing_bins == "left_margin_0_50kb")
domain_end   <- which(existing_bins == "right_margin_0_50kb")

# ----------------------------------------------------------------------------
# Build plot
# ----------------------------------------------------------------------------
p_metaprofile <- ggplot(
  domain_meta,
  aes(
    x = bin_factor,
    y = mean_signal,
    color = sample,
    fill = sample,
    group = sample
  )
) +

  # Shaded domain interior
  annotate(
    "rect",
    xmin = domain_start - 0.5,
    xmax = domain_end + 0.5,
    ymin = -Inf,
    ymax = Inf,
    fill = "grey70",
    alpha = 0.12
  ) +

  # Zero line
  geom_hline(
    yintercept = 0,
    linetype = "dashed",
    linewidth = 0.4,
    colour = "grey45"
  ) +

  # Error ribbons first
  geom_ribbon(
    aes(
      ymin = mean_signal - se_signal,
      ymax = mean_signal + se_signal
    ),
    alpha = 0.10,
    linewidth = 0,
    show.legend = FALSE
  ) +

  # Lines and points on top
  geom_line(
    linewidth = 0.95
  ) +
  geom_point(
    size = 2.0
  ) +

  # Manual colors
  scale_color_manual(
    values = SAMPLE_COLORS,
    labels = SAMPLE_LABELS,
    guide = guide_legend(
      nrow = 1,
      byrow = TRUE,
      override.aes = list(
        linewidth = 1.1,
        shape = 16,
        size = 3
      )
    )
  ) +
  scale_fill_manual(
    values = SAMPLE_COLORS,
    labels = SAMPLE_LABELS,
    guide = "none"
  ) +

  # Cleaner y-axis
  scale_y_continuous(
    breaks = breaks_pretty(n = 5),
    expand = expansion(mult = c(0.03, 0.06))
  ) +

  labs(
    x = "Distance from CENP-A enriched satellite array",
    y = expression("Mean log"[2] * "(normalized CUT&Tag signal)"),
    color = NULL,
    title = "CENP-A signal centered on merged CENP-A enriched satellite arrays",
    subtitle = paste0(
      "+/- 1 Mb flanks; n = ", n_domains_total,
      " domains. Shaded region denotes the domain interior."
    )
  ) +

  theme_cenpa +

  theme(
    # Titles
    plot.title = element_text(
      size = 12,
      face = "bold",
      hjust = 0,
      margin = margin(b = 3)
    ),
    plot.subtitle = element_text(
      size = 9,
      colour = "grey30",
      hjust = 0,
      margin = margin(b = 10)
    ),

    # Axes
    axis.title.x = element_text(
      size = 10,
      margin = margin(t = 10)
    ),
    axis.title.y = element_text(
      size = 10,
      margin = margin(r = 10)
    ),
    axis.text.x = element_text(
      angle = 40,
      hjust = 1,
      vjust = 1,
      size = 8
    ),
    axis.text.y = element_text(
      size = 8.5,
      colour = "black"
    ),

    # Gridlines: keep horizontal, drop vertical clutter
    panel.grid.major.x = element_blank(),
    panel.grid.minor = element_blank(),
    panel.grid.major.y = element_line(
      colour = "grey88",
      linewidth = 0.35
    ),

    # Border
    panel.border = element_rect(
      colour = "grey35",
      fill = NA,
      linewidth = 0.55
    ),

    # Legend
    legend.position = "top",
    legend.justification = "left",
    legend.direction = "horizontal",
    legend.text = element_text(size = 9),
    legend.key.width = unit(0.8, "cm"),
    legend.key.height = unit(0.35, "cm"),
    legend.margin = margin(b = 4),

    # Margins
    plot.margin = margin(
      t = 8,
      r = 10,
      b = 6,
      l = 8
    )
  )

print(p_metaprofile)

# ----------------------------------------------------------------------------
# Export (recommended landscape size)
# ----------------------------------------------------------------------------
ggsave(
  filename = file.path("CENPA_domain_centered_metaprofile.pdf"),
  plot = p_metaprofile,
  width = 8.8,
  height = 5.6,
  units = "in",
  device = cairo_pdf
)

ggsave(
  filename = file.path("CENPA_domain_centered_metaprofile.png"),
  plot = p_metaprofile,
  width = 8.8,
  height = 5.6,
  units = "in",
  dpi = 600,
  bg = "white"
)

---
## Per-domain Enrichment vs Local Flank Background

Each point = one CENP-A enriched satellite array (merged centroAnno intervals within 250 kb). x-axis = chromosome-pooled local flank background; y-axis = domain foreground signal. Points above the diagonal indicate enrichment. Color = chromosome classification by CENP-A enrichment magnitude.</text>

In [ ]:
# ============================================================================
# Per-domain enrichment vs local flank background
# ============================================================================

# Per-domain foreground signal
domain_fg <- domain_counts[, .(
  fg_median = median(log2_signal, na.rm = TRUE)
), by = .(sample, chrom, start, end, domain_id)]

# Pooled flank background per sample per chromosome
domain_bg_by_chr <- domain_flanks[, .(
  bg_median = median(log2_signal, na.rm = TRUE),
  bg_mean = mean(log2_signal, na.rm = TRUE),
  n_flanks = .N
), by = .(sample, chrom)]

# Match domains to chromosome-level background
domain_compare <- merge(domain_fg, domain_bg_by_chr, by = c("sample", "chrom"))
domain_compare[, delta := fg_median - bg_median]

# CENP-A rep1 for scatter
d_plot_150 <- domain_compare[sample == "XG_150"]
chr_class <- chr_classification[, .(chrom, category)]
d_plot_150 <- merge(d_plot_150, chr_class, by = "chrom", all.x = TRUE)

x_range <- range(d_plot_150$bg_median, na.rm = TRUE)
y_range <- range(d_plot_150$fg_median, na.rm = TRUE)
xy_limits <- c(min(x_range[1], y_range[1]), max(x_range[2], y_range[2]))

pct_above <- round(mean(d_plot_150$fg_median > d_plot_150$bg_median) * 100)

p_domain_scatter <- ggplot(d_plot_150, aes(x = bg_median, y = fg_median, color = category)) +
  geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "grey50") +
  geom_point(alpha = 0.7, size = 1.5) +
  scale_color_manual(values = cat_colors, name = "Chromosome\nclassification") +
  coord_fixed(xlim = xy_limits, ylim = xy_limits) +
  labs(x = expression("Chr-pooled flank median log"[2] * "(normalized signal)"),
       y = expression("Domain median log"[2] * "(normalized signal)"),
       title = "Per-domain CENP-A enrichment vs local flank background",
       subtitle = paste0("CENP-A rep1; ", nrow(d_plot_150), " domains; ",
                         pct_above, "% above diagonal")) +
  theme_cenpa

print(p_domain_scatter)

cat("\nDomain-level statistics:\n")
domain_wide <- dcast(domain_compare[sample %in% c("XG_150", "XG_151")],
                     chrom + domain_id ~ sample, value.var = "delta")
n_pos <- sum(domain_wide$XG_150 > 0, na.rm = TRUE)
n_tot <- sum(!is.na(domain_wide$XG_150))
cat(sprintf("  Domains above chr-pooled flank (rep1): %d/%d (%.1f%%)\n",
            n_pos, n_tot, 100 * n_pos / n_tot))

---
## TRF Period-Size Stratified CENP-A Enrichment

**Biological question:** Is CENP-A binding specific to centromeric repeat periods
(348-390 bp), or does it occur at all tandem repeats regardless of monomer size?

**Approach:** All TRF-detected tandem repeats genome-wide were binned by monomer
period length (9 bins, from 1-10 bp microsatellites to 391+ bp), isolating the
monomer families of interest (193-195, 348-349, 386-390 bp) as their own bins.
Adjacent same-period intervals were merged into repeat arrays (bedtools
merge -d 0) and CENP-A and H3K27ac CUT&Tag signal was quantified at each merged
array. The merged repeat array is the unit of observation, removing the
non-independence of overlapping intervals within shared arrays. Every bin is
tested against a chromosome- and length-matched shuffled null (1,000
iterations): empirical P values compare the observed foreground median against
the per-iteration median null distribution.

**Bins (n = merged repeat arrays):**
1. 1-10 bp (microsatellites; n = 721,950) — negative control
2. 11-50 bp (minisatellites; n = 344,459) — negative control
3. 51-192 bp (n = 29,673)
4. 193-195 bp (n = 12,631) — enriched CENP-A monomer family
5. 196-347 bp (n = 7,345)
6. 348-349 bp (n = 799) — dominant degu centromeric satellite; CENP-A DEPLETED
7. 350-385 bp (n = 233)
8. 386-390 bp (n = 883) — enriched CENP-A monomer family
9. 391+ bp (n = 823)

**Null design:** Bins 1-5 (many arrays) draw a fresh seeded 5,000-array
subsample per iteration, then perform one chromosome-/length-matched shuffle per
array — the null marginalizes over both subsampling and shuffle randomness.
Bins 6-9 (tiny, <= 900 arrays) shuffle the full array set every iteration.
In both cases the null = mean of the 1,000 per-iteration medians.

**Primary inference:** Permutation test across all 9 bins. **Secondary:**
Kruskal-Wallis across all bins.

**Script:** `period-enrichment/scripts/3_analyze_period_enrichment_array.R`

**Environment:** `conda activate r-visualizations`


---
**Companion barplots: per-bin merged repeat arrays (context for the enrichment violin).** Left panel: **number of merged repeat arrays** per TRF period bin. Right panel: **mean array length** (bp). Both x-axes are log₁₀ — counts span 233 to 721,950 and mean lengths 60 bp to 142 kb, so a linear axis would squash the small bins into slivers. The bars use the same sampling unit as the enrichment violin (merged repeat arrays, bedtools merge -d 0 within each period bin), and the 9 bins are stacked in the same order (1–10 bp at the bottom, 391+ bp at the top), so the strip lines up row-for-row when placed to the left of the violin.

Composition story: the negative-control micro/minisatellite bins 1–2 dominate by **count** (722k / 344k short arrays), while the 348–349 bp centromeric-satellite bin 6 has the fewest arrays (799) but by far the **longest** (mean ~142 kb) — the degu centromere forms long tandem arrays. The enriched monomer families sit in between (193–195 bp: 12,631 arrays, mean 838 bp; 386–390 bp: 883 arrays, mean 2.4 kb). The per-bin n counts previously shown on the violin's y-axis now live in the left panel.

**Script:** `period-enrichment/scripts/period_array_count_length.R` (reads `data/merged/arrays/merged_arrays_all_bins.bed`, caches per-bin stats to `results/period_array_count_length.tsv`). **Environment:** `conda activate r-visualizations`


In [ ]:
PERIOD_PLOTS

In [ ]:
# ============================================================================
# Period enrichment: companion barplots (n merged arrays + mean array length)
#
# Two horizontal barplots sharing the violin's 9-bin order (1-10 bp at the
# bottom, 391+ bp at the top), sized to sit to the LEFT of the enrichment
# violin in the assembled figure. Plotting code lives in the standalone R
# script:
#   period-enrichment/scripts/period_array_count_length.R
# which accepts optional WIDTH HEIGHT (inches) arguments, so re-saving at a
# custom size is fast (seconds). Set the FILE-SIZE knobs below, then re-run
# this cell. (Run once -> also overwrites the saved .pdf/.svg files.)
# ============================================================================

PERIOD_SCRIPTS <- file.path(BASE_DIR, "period-enrichment", "scripts")
PERIOD_PLOTS   <- file.path(BASE_DIR, "period-enrichment", "plots")
strip_script   <- file.path(PERIOD_SCRIPTS, "period_array_count_length.R")
rscript_bin    <- "/tscc/projects/ps-renlab2/jhc103/miniconda3-storage/envs/r-visualizations/bin/Rscript"

# File size (inches) - adjust as needed, then re-run this cell.
#   at 300 dpi, PNG pixels = inches x 300 (e.g. 7.5 x 9 in -> 2250 x 2700 px)
strip_out_width  <- 7.5
strip_out_height <- 9

if (file.exists(strip_script) && file.exists(rscript_bin)) {
  message(sprintf("Regenerating barplot strip at %g x %g in ...",
                  strip_out_width, strip_out_height))
  system2(rscript_bin, c(strip_script, strip_out_width, strip_out_height))
} else {
  message("Strip script or Rscript not found - displaying cached figure if any.")
}

strip_fig <- file.path(PERIOD_PLOTS, "period_array_count_length.png")
if (file.exists(strip_fig)) {
  if (requireNamespace("IRdisplay", quietly = TRUE)) {
    IRdisplay::display_png(file = strip_fig)
  }
} else {
  message("Could not generate the strip figure.")
}


---
**Figure annotation.** Violins show the distribution of pseudocount-adjusted log₂ CUT&Tag signal across observed merged repeat arrays, centered by subtracting the permutation-derived median null baseline for each repeat-period bin and replicate (Δ = log₂(CPM+p) − null_median). The null is the mean of per-iteration medians over 1,000 chromosome- and length-matched shuffles: bins 1–5 use a fresh 5,000-array subsample per iteration; bins 6–9 shuffle the full set. White circles mark the median per-array enrichment score per replicate. Positive Δ indicates signal above the typical shuffled background; negative indicates below — i.e. "above/below null baseline", not a raw-CPM fold change. Significance stars denote empirical permutation tests on matched bin-level medians (paired 5,000-array resampling for bins 1–5); they are not derived from the individual array values shown. Independence: adjacent same-period intervals were merged into repeat arrays (bedtools merge -d 0), and the merged array is the sampling unit for both the observed and the shuffled null — so per-bin n counts independent arrays rather than overlapping intervals.


In [ ]:
# ============================================================================
# Period enrichment: Horizontal violin plot (CENP-A signal by TRF period)
#
# Displays the pre-computed MERGED-ARRAY figure from period_violin_array.R.
# ============================================================================

PERIOD_PLOTS <- file.path(BASE_DIR, "period-enrichment", "plots")
PERIOD_RESULTS <- file.path(BASE_DIR, "period-enrichment", "results")

violin_fig <- file.path(PERIOD_PLOTS, "period_violin_cenpa_merged.png")

if (!file.exists(violin_fig)) {
  message("Violin figure not yet generated. Run:")
  message("  cd period-enrichment && Rscript scripts/period_violin_array.R")
} else {
  if (requireNamespace("IRdisplay", quietly = TRUE)) {
    IRdisplay::display_png(file = violin_fig)
  }
}

In [ ]:
# ============================================================================
# Period enrichment: Horizontal violin plot (H3K27ac signal by TRF period)
#
# Same figure as the CENP-A violin, for the H3K27ac replicates (XG_152/153),
# from the same parameterized script:
#   period-enrichment/scripts/period_violin_array.R H3K27ac
# ============================================================================

violin_h3k27ac_fig <- file.path(PERIOD_PLOTS, "period_violin_h3k27ac_merged.png")

if (file.exists(violin_h3k27ac_fig)) {
  if (requireNamespace("IRdisplay", quietly = TRUE)) {
    IRdisplay::display_png(file = violin_h3k27ac_fig)
  }
} else {
  message("H3K27ac violin figure not yet generated. Run:")
  message("  cd period-enrichment && Rscript scripts/period_violin_array.R H3K27ac")
}

In [ ]:
# ============================================================================
# Period enrichment: All-bins overview (CENP-A vs H3K27ac) — MERGED arrays
# ============================================================================

overview_fig <- file.path(PERIOD_PLOTS, "period_all_bins_overview_merged.png")

if (file.exists(overview_fig)) {
  if (requireNamespace("IRdisplay", quietly = TRUE)) {
    IRdisplay::display_png(file = overview_fig)
  }
} else {
  message("Overview figure not yet available.")
}

In [ ]:
# ============================================================================
# Period enrichment: Permutation test results table (MERGED arrays)
# ============================================================================

perm_results_file <- file.path(PERIOD_RESULTS, "period_enrichment_merged_permutation_results.csv")

if (file.exists(perm_results_file)) {
  library(data.table)
  perm_res <- fread(perm_results_file)

  # Pretty-print CENP-A results
  cat("\n=== Permutation Test: CENP-A Enrichment by TRF Period (merged arrays) ===\n")
  cat("Empirical P: fraction of 1,000 shuffles with bg >= fg median\n")
  cat("FDR: Benjamini-Hochberg correction per sample\n\n")

  for (s in c("XG_150", "XG_151")) {
    cat(sprintf("\n--- %s ---\n", s))
    sub <- perm_res[sample == s][order(bin_id)]
    cat(sprintf("%-6s %-25s %8s %8s %8s %8s %8s %6s\n",
                "Bin", "Label", "Fg_med", "Bg_null", "Bg_SD", "Z", "P_enr", "P_adj"))
    cat(strrep("-", 77), "\n")
    for (i in seq_len(nrow(sub))) {
      r <- sub[i]
      cat(sprintf("%-6s %-25s %8.3f %8.3f %8.3f %8.2f %8.4f %8.4f %s\n",
                  r$bin_id, substring(r$bin_label, 1, 25),
                  r$fg_median, r$bg_mean_of_medians, r$bg_sd_of_medians,
                  r$z_score, r$p_enrich, r$p_adj, r$sig))
    }
  }

  # Summary for text
  cat("\n=== Key Findings ===\n")
  for (s in c("XG_150", "XG_151")) {
    sub <- perm_res[sample == s]
    sig_bins <- sub[p_adj < 0.05, bin_id]
    cat(sprintf("%s: Significant bins (FDR<0.05): %s\n",
                s, paste(sig_bins, collapse = ", ")))
  }
} else {
  message("Merged permutation results not yet available. Run 3_analyze_period_enrichment_array.R first.")
}

---
**Z-score heatmap (per chromosome x period bin; rep1, merged arrays).** Z = (observed median − null mean) / null SD, capped at ±5. **Red = enriched (Z>0), Blue = depleted (Z<0).** A star inside a cell means that *chromosome's* bin is significantly enriched/depleted (* p_adj<0.05, ** p_adj<0.01, *** p_adj<0.001; per-chromosome permutation test, BH-FDR within each bin). Grey cells = no 348-349 bp (bin 6) arrays on that chromosome (chr13, chr14, chr19, chr23); chrY has no bin 3-9 arrays. Note: per-chromosome tests are low-power in sparse bins (e.g. bin 6), where the shuffled null is built from the same few arrays.
---

In [ ]:
# ============================================================================
# Period enrichment: Supplementary figures
# ============================================================================

# Z-score heatmap per chromosome per bin (MERGED arrays)
# ---------------------------------------------------------------------------
# Regenerate the heatmap at a CUSTOM SIZE from inside the notebook, then
# display it. Plotting code lives in the standalone R script:
#   period-enrichment/scripts/supp_permutation_zscore_heatmap_merged.R
# which accepts optional WIDTH HEIGHT args (inches, default 11 8) and reads
# the cached Z-score matrix, so regeneration is fast (seconds). Set the two
# FILE-SIZE knobs below, then re-run this cell.
# ---------------------------------------------------------------------------

PERIOD_SCRIPTS <- file.path(BASE_DIR, "period-enrichment", "scripts")
zheat_script <- file.path(PERIOD_SCRIPTS,
                          "supp_permutation_zscore_heatmap_merged.R")
rscript_bin  <- "/tscc/projects/ps-renlab2/jhc103/miniconda3-storage/envs/r-visualizations/bin/Rscript"

# File size (inches) - adjust as needed, then re-run this cell.
#   at 300 dpi, PNG pixels = inches x 300 (e.g. 11 x 8 in -> 3300 x 2400 px)
zheat_out_width  <- 6.4
zheat_out_height <- 8

if (file.exists(zheat_script) && file.exists(rscript_bin)) {
  message(sprintf("Regenerating heatmap at %g x %g in ...",
                  zheat_out_width, zheat_out_height))
  system2(rscript_bin, c(zheat_script, zheat_out_width, zheat_out_height))
} else {
  message("Heatmap script or r-visualizations Rscript not found.")
}

# Display size (px) - adjust as needed
zheat_fig <- file.path(PERIOD_PLOTS, "supp_permutation_zscore_heatmap_merged.png")
zheat_width <- 1200
zheat_height <- 800
if (file.exists(zheat_fig)) {
  if (requireNamespace("IRdisplay", quietly = TRUE)) {
    IRdisplay::display_png(file = zheat_fig,
                           width = zheat_width, height = zheat_height)
  }
} else {
  message("Z-score heatmap not yet available.")
}


---
**Observed vs null per-array distribution for the CENP-A monomer families (merged arrays).** Grey = per-array signal at chromosome- and length-matched shuffled placements of merged repeat arrays (1,000 iterations, pooled). Colored = observed arrays; the dashed colored line = observed median (descriptive). The spike at log2 = −9.4 is the zero-coverage mass. Direction labels (enriched/depleted) and P_adj < 0.001 come from the merged-array permutation test of iteration-specific matched observed-vs-shuffled median differences — not from these curves.
---

In [ ]:
# ============================================================================
# Monomer families (bins 4/6/8): observed vs null per-array distribution
# ----------------------------------------------------------------------------
# MERGED-ARRAY version — one value per merged repeat array (bedtools merge -d 0
# within each period bin). Plotting code lives in the standalone R script:
#   period-enrichment/scripts/supp_monomer_bins_foreground_vs_null_overlay_array.R
#
# Regenerate the figure (in a shell / terminal):
#   conda activate r-visualizations
#   Rscript period-enrichment/scripts/supp_monomer_bins_foreground_vs_null_overlay_array.R
# ============================================================================

null_fig <- file.path(PERIOD_PLOTS, "supp_monomer_bins_foreground_vs_null_distribution_merged.png")
if (file.exists(null_fig)) {
  if (requireNamespace("IRdisplay", quietly = TRUE)) {
    IRdisplay::display_png(file = null_fig)
  }
} else {
  message("Merged monomer-bins figure not yet generated. Run the R script above.")
}

---
**Supplementary figure — the 195 bp tandem repeats are L1 retrotransposon sequence, not a satellite.** (a) RepeatMasker class composition of the TRF period-binned merged arrays. The **195 bp arrays** (bin 4; 12,631 arrays, 10.6 Mb) are **99.4% LINE/L1**, dominated by degu L1 families `rnd-1_family-189` (92.3%) and `rnd-1_family-18` (7.0%). The **349 bp arrays** (bin 6; 799 arrays, 113.3 Mb) are **99.3% unannotated** ("Unknown" = no RepeatMasker library match) — a lineage-specific satellite, consistent with 349 bp being the degu centromere. (b) The 195 bp unit is the **5′-terminal tandem repeat of the degu L1**: running TRF with the same parameters as the genome-wide call on the L1 consensus finds 4.1 copies of a 195 bp unit at the 5′ end of `rnd-1_family-189` (2.2 copies in `rnd-1_family-18`). (c) Example locus chr21:83.26–83.34 Mb — the longest 195 bp array (71.9 kb) is a **tandem array of L1 fragments**, a repeating unit of [L1 rnd-1_family-189 → rnd-1_family-18 → LTR/other → Unknown junctions]; the TRF 195 bp arrays (black) span these L1-rich regions. Conclusion: the "195/389 bp satellite" is L1 sequence (389 bp = 2×195 bp dimer at the same L1 loci) and should be described as such in the paper, not as a centromere-related satellite.

In [ ]:
# ============================================================================
# Supplementary figure: the 195 bp repeats are L1 retrotransposon sequence
#
# Displays the 3-panel figure from 195bp-repeatmasker-overlap/195bp_l1_overlap.R
# (data tables cached by prepare_panel_data.py).
# ============================================================================

L1_PLOTS <- file.path(BASE_DIR, "cenpa-repeat-chromosome")
l1_fig <- file.path(L1_PLOTS, "195bp_l1_overlap.png")

if (!file.exists(l1_fig)) {
  message("Figure not yet generated. Run:")
  message("  cd 195bp-repeatmasker-overlap && Rscript prepare_panel_data.py")
  message("  cd 195bp-repeatmasker-overlap && Rscript 195bp_l1_overlap.R")
} else {
  if (requireNamespace("IRdisplay", quietly = TRUE)) {
    IRdisplay::display_png(file = l1_fig)
  }
}


---
## Summary

### Key Results

| Metric | Value |
|--------|-------|
| Chromosomes with positive enrichment (Δ > 0) | **30/30** |
| Chromosomes FDR < 0.05 (BH-corrected empirical P) | **see per-chromosome table** |
| CENP-A replicate Spearman rho | **0.823** |
| H3K27ac replicate Spearman rho | **0.977** |
| Domains above chr-pooled flank | **93.7%** |
| Enrichment magnitude range (Δ log₂ normalized signal) | **varies by chromosome** |
| Shuffle iterations per background estimate | **auto-detected from data** |

### Statistical Framework

The chromosome-level analysis uses chromosome- and length-matched shuffle iterations. For each chromosome c:
- **Observed** O_c = median log₂(normalized CUT&Tag signal) across domains on chromosome c
- **Expected** median_i(B_{c,i}) = median across all shuffle iterations of the per-iteration median shuffled signal
- **Enrichment** Δ_c = O_c − median_i(B_{c,i})
- **Fold enrichment** = 2^{Δ_c}
- **Empirical P** = (1 + count of B_{c,i} ≥ O_c) / (N_iterations + 1), with Benjamini-Hochberg correction across 30 chromosomes

This replaces the original single-iteration background (Bg1 iter_001) with a statistically robust null distribution, avoiding dependence on a single random shuffle.

### Chromosome Classification

All 30 chromosomes show CENP-A enrichment above shuffled background. Classification by **magnitude** (tertiles of mean Δ across replicates):
- **Strong enrichment** (top third): ~10 chromosomes
- **Moderate enrichment** (middle third): ~10 chromosomes
- **Weak enrichment** (bottom third): ~10 chromosomes

chr1 has lower-magnitude enrichment but is still enriched above background — consistent with its smaller, more distributed CENP-A enriched satellite array architecture.

### Data type and normalization

CUT&Tag (Cleavage Under Targets & Tagmentation) is an **epigenomic chromatin profiling assay** that maps protein–DNA interactions in situ. Unlike RNA-seq, which measures transcript abundance, CUT&Tag directly localizes chromatin-bound proteins (here CENP-A, a centromere-specific histone H3 variant, and H3K27ac, a histone modification marking active chromatin).

Signal is quantified as **paired-end CUT&Tag fragment counts** per genomic interval, normalized by interval length and library size (fragments per kilobase per million mapped). The resulting normalized signal values are then log₂-transformed with a pseudocount for visualization and statistical testing. We use the term **"normalized CUT&Tag signal"** throughout — deliberately avoiding "FPKM" and "expression" terminology, which implies a transcript-centric measurement model inappropriate for chromatin enrichment data.

### Why are log₂(normalized signal) values negative?

log₂(x) < 0 whenever x < 1. At individual 1-kb bins, normalized fragment density is typically below 1 even within enriched domains. This is normal for sparse CUT&Tag data. The relevant metric is the **Δ** between domain and background, not the absolute sign of the log-transformed values.

**Last updated:** 2026-07-22